Raw data

In [1]:
customers = [
    {
        "customer_id": "C001",
        "name": "  RAJ ",
        "age": "28",
        "income": "55000.50",
        "city": " CHENNAI ",
        "credit_score": "760",
        "purchased": "yes"
    },
    {
        "customer_id": "C002",
        "name": "Anita",
        "age": "",
        "income": "62000",
        "city": "BENGALURU",
        "credit_score": "invalid",
        "purchased": "no"
    },
    {
        "customer_id": "C003",
        "name": "  Kumar",
        "age": "42",
        "income": None,
        "city": " chennai",
        "credit_score": "680",
        "purchased": "YES"
    },
    {
        "customer_id": "C004",
        "name": "",
        "age": "17",
        "income": "-5000",
        "city": None,
        "credit_score": "820",
        "purchased": "unknown"
    }
]

Cleaning functions

In [2]:
def clean_text(
    value,
    default="unknown",
    case="lower"
):
    if value is None:
        return default

    cleaned_value = str(value).strip()

    if cleaned_value == "":
        return default

    if case == "lower":
        return cleaned_value.lower()

    if case == "upper":
        return cleaned_value.upper()

    if case == "title":
        return cleaned_value.title()

    return cleaned_value


def safe_int(value):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return None


def safe_float(value):
    try:
        return float(str(value).strip())
    except (TypeError, ValueError):
        return None


def clean_boolean(value):
    if isinstance(value, bool):
        return value

    if value is None:
        return None

    cleaned_value = str(value).strip().lower()

    true_values = {"yes", "y", "true", "1"}
    false_values = {"no", "n", "false", "0"}

    if cleaned_value in true_values:
        return True

    if cleaned_value in false_values:
        return False

    return None

Feature engineering

In [3]:
def assign_income_band(income):
    if income is None:
        return "unknown"

    if income < 40_000:
        return "low"

    if income < 70_000:
        return "medium"

    return "high"


def assign_credit_risk(credit_score):
    if credit_score is None:
        return "unknown"

    if credit_score >= 750:
        return "low"

    if credit_score >= 650:
        return "medium"

    return "high"

Validation

In [4]:
def validate_customer(customer):
    errors = []

    if customer["customer_id"] == "":
        errors.append("Customer ID cannot be empty.")

    if customer["name"] == "Unknown":
        errors.append("Customer name cannot be empty.")

    age = customer["age"]

    if age is None:
        errors.append("Age is missing or invalid.")
    elif not 18 <= age <= 100:
        errors.append("Age must be between 18 and 100.")

    income = customer["income"]

    if income is None:
        errors.append("Income is missing or invalid.")
    elif income < 0:
        errors.append("Income cannot be negative.")

    credit_score = customer["credit_score"]

    if credit_score is None:
        errors.append("Credit score is missing or invalid.")
    elif not 300 <= credit_score <= 900:
        errors.append(
            "Credit score must be between 300 and 900."
        )

    if customer["purchased"] is None:
        errors.append(
            "Purchased value is missing or invalid."
        )

    return errors

Process one customer

In [5]:
def preprocess_customer(raw_customer):
    cleaned_customer = {
        "customer_id": clean_text(
            raw_customer.get("customer_id"),
            default="",
            case="upper"
        ),
        "name": clean_text(
            raw_customer.get("name"),
            default="Unknown",
            case="title"
        ),
        "age": safe_int(
            raw_customer.get("age")
        ),
        "income": safe_float(
            raw_customer.get("income")
        ),
        "city": clean_text(
            raw_customer.get("city"),
            default="unknown",
            case="lower"
        ),
        "credit_score": safe_int(
            raw_customer.get("credit_score")
        ),
        "purchased": clean_boolean(
            raw_customer.get("purchased")
        )
    }

    cleaned_customer["income_band"] = (
        assign_income_band(
            cleaned_customer["income"]
        )
    )

    cleaned_customer["credit_risk"] = (
        assign_credit_risk(
            cleaned_customer["credit_score"]
        )
    )

    errors = validate_customer(cleaned_customer)

    cleaned_customer["is_valid"] = len(errors) == 0
    cleaned_customer["errors"] = errors

    return cleaned_customer

Process all customers

In [6]:
def preprocess_customers(raw_customers):
    return [
        preprocess_customer(customer)
        for customer in raw_customers
    ]


cleaned_customers = preprocess_customers(customers)

for customer in cleaned_customers:
    print(customer)
    print()

{'customer_id': 'C001', 'name': 'Raj', 'age': 28, 'income': 55000.5, 'city': 'chennai', 'credit_score': 760, 'purchased': True, 'income_band': 'medium', 'credit_risk': 'low', 'is_valid': True, 'errors': []}

{'customer_id': 'C002', 'name': 'Anita', 'age': None, 'income': 62000.0, 'city': 'bengaluru', 'credit_score': None, 'purchased': False, 'income_band': 'medium', 'credit_risk': 'unknown', 'is_valid': False, 'errors': ['Age is missing or invalid.', 'Credit score is missing or invalid.']}

{'customer_id': 'C003', 'name': 'Kumar', 'age': 42, 'income': None, 'city': 'chennai', 'credit_score': 680, 'purchased': True, 'income_band': 'unknown', 'credit_risk': 'medium', 'is_valid': False, 'errors': ['Income is missing or invalid.']}

{'customer_id': 'C004', 'name': 'Unknown', 'age': 17, 'income': -5000.0, 'city': 'unknown', 'credit_score': 820, 'purchased': None, 'income_band': 'low', 'credit_risk': 'low', 'is_valid': False, 'errors': ['Customer name cannot be empty.', 'Age must be between 

Summary functions

In [7]:
def calculate_average(values):
    valid_values = [
        value
        for value in values
        if value is not None
    ]

    if not valid_values:
        return None

    return sum(valid_values) / len(valid_values)


def count_categories(values):
    counts = {}

    for value in values:
        counts[value] = counts.get(value, 0) + 1

    return counts

Generate summary

In [8]:
def calculate_customer_summary(customers):
    total_records = len(customers)

    valid_records = sum(
        1
        for customer in customers
        if customer["is_valid"]
    )

    invalid_records = total_records - valid_records

    valid_ages = [
        customer["age"]
        for customer in customers
        if customer["age"] is not None
        and 18 <= customer["age"] <= 100
    ]

    valid_incomes = [
        customer["income"]
        for customer in customers
        if customer["income"] is not None
        and customer["income"] >= 0
    ]

    known_purchase_values = [
        customer["purchased"]
        for customer in customers
        if customer["purchased"] is not None
    ]

    purchased_count = sum(
        value is True
        for value in known_purchase_values
    )

    purchase_rate = (
        purchased_count / len(known_purchase_values)
        if known_purchase_values
        else 0
    )

    return {
        "total_records": total_records,
        "valid_records": valid_records,
        "invalid_records": invalid_records,
        "average_age": calculate_average(valid_ages),
        "average_income": calculate_average(valid_incomes),
        "purchased_count": purchased_count,
        "purchase_rate": purchase_rate,
        "city_counts": count_categories(
            [
                customer["city"]
                for customer in customers
            ]
        ),
        "credit_risk_counts": count_categories(
            [
                customer["credit_risk"]
                for customer in customers
            ]
        ),
        "income_band_counts": count_categories(
            [
                customer["income_band"]
                for customer in customers
            ]
        )
    }


summary = calculate_customer_summary(
    cleaned_customers
)

for key, value in summary.items():
    print(f"{key}: {value}")

total_records: 4
valid_records: 1
invalid_records: 3
average_age: 35.0
average_income: 58500.25
purchased_count: 2
purchase_rate: 0.6666666666666666
city_counts: {'chennai': 2, 'bengaluru': 1, 'unknown': 1}
credit_risk_counts: {'low': 2, 'unknown': 1, 'medium': 1}
income_band_counts: {'medium': 2, 'unknown': 1, 'low': 1}


Export results

In [ ]:
import json


valid_customers = [
    customer
    for customer in cleaned_customers
    if customer["is_valid"]
]

invalid_customers = [
    customer
    for customer in cleaned_customers
    if not customer["is_valid"]
]

with open(
    "cleaned_customers.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        cleaned_customers,
        file,
        indent=4
    )

with open(
    "valid_customers.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        valid_customers,
        file,
        indent=4
    )

with open(
    "invalid_customers.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        invalid_customers,
        file,
        indent=4
    )

with open(
    "customer_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        indent=4
    )

print("All JSON files exported successfully")

Tests

In [9]:
assert safe_int("28") == 28
assert safe_int("") is None
assert safe_int("invalid") is None
assert safe_float("55000.50") == 55000.50
assert safe_float(None) is None

assert clean_boolean("YES") is True
assert clean_boolean("no") is False
assert clean_boolean("unknown") is None

assert clean_text(" CHENNAI ") == "chennai"
assert clean_text(" raj ", case="title") == "Raj"
assert clean_text(None) == "unknown"

assert assign_income_band(30_000) == "low"
assert assign_income_band(55_000) == "medium"
assert assign_income_band(90_000) == "high"

assert assign_credit_risk(800) == "low"
assert assign_credit_risk(700) == "medium"
assert assign_credit_risk(600) == "high"

valid_test_customer = {
    "customer_id": "C100",
    "name": "  rajasekar ",
    "age": "30",
    "income": "75000",
    "city": " CHENNAI ",
    "credit_score": "780",
    "purchased": "yes"
}

result = preprocess_customer(valid_test_customer)

assert result["name"] == "Rajasekar"
assert result["age"] == 30
assert result["income"] == 75000.0
assert result["city"] == "chennai"
assert result["purchased"] is True
assert result["income_band"] == "high"
assert result["credit_risk"] == "low"
assert result["is_valid"] is True
assert result["errors"] == []

print("All preprocessing tests passed")

All preprocessing tests passed
